# PARTE 1 — Modelado Relacional + JSON Jerárquico (corregido)

**Sistema:** `GestionAeropuerto` · **Rango:** 2023-01-01 → 2024-12-31 (731 días)

Correcciones aplicadas vs. versión preliminar:
- JSON cubre los 731 días del rango (antes solo enero 2023).
- Mismo universo de datos en relacional y jerárquico.
- `Tripulacion_Vuelo` poblada (P4 la usa para traversal multinivel).
- `fecha_nacimiento_iso` en lugar de `edad` precomputada (determinista).
- Fechas serializadas como ISO `YYYY-MM-DDTHH:MM:SS` (P3 BigQuery las ingiere tal cual).
- Semillas fijas (Python + PG) para reproducibilidad.

In [2]:
import os, json, random, string, time, csv
from datetime import datetime, date
from decimal import Decimal
from collections import defaultdict

import psycopg2
from psycopg2.extras import execute_values

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

DB_CONFIG = {'host': 'localhost', 'port': 5432, 'user': 'jesusrodarte'}
DB_NAME   = 'gestionaeropuerto'

FECHA_INICIO = datetime(2023, 1, 1)
FECHA_FIN    = datetime(2024, 12, 31)
VUELOS_POR_DIA  = (5, 7)
PASAJEROS_VUELO = (40, 80)
TRIPULACION_POR_VUELO = (2, 3)

OUTPUT_DIR = './parte1'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}  |  Rango: {FECHA_INICIO.date()} → {FECHA_FIN.date()}')

Output: ./parte1  |  Rango: 2023-01-01 → 2024-12-31


## 1. Reset limpio de la BD

In [3]:
conn_admin = psycopg2.connect(dbname='postgres', **DB_CONFIG)
conn_admin.autocommit = True
cur = conn_admin.cursor()
cur.execute("SELECT pg_terminate_backend(pid) FROM pg_stat_activity WHERE datname=%s AND pid<>pg_backend_pid()", (DB_NAME,))
cur.execute(f"DROP DATABASE IF EXISTS {DB_NAME}")
cur.execute(f"CREATE DATABASE {DB_NAME}")
cur.close(); conn_admin.close()
print(f'BD {DB_NAME}: recreada')

BD gestionaeropuerto: recreada


## 2. DDL — 13 tablas (catálogos + transaccionales + N:M)

In [4]:
DDL = """
CREATE TABLE Aerolineas (
    Codigo_IATA_Aerolinea CHAR(2) PRIMARY KEY,
    Nombre_Comercial      VARCHAR(100) NOT NULL,
    Pais_Origen           VARCHAR(50)  NOT NULL
);
CREATE TABLE Aeropuertos (
    Codigo_IATA_Aeropuerto CHAR(3) PRIMARY KEY,
    Nombre_Oficial         VARCHAR(150) NOT NULL,
    Ciudad                 VARCHAR(100) NOT NULL,
    Pais                   VARCHAR(50)  NOT NULL
);
CREATE TABLE Modelos_Avion (
    Codigo_Modelo       VARCHAR(15) PRIMARY KEY,
    Fabricante          VARCHAR(50) NOT NULL,
    Capacidad_Pasajeros INT NOT NULL,
    Alcance_Km          INT
);
CREATE TABLE Empleados (
    Numero_Licencia       VARCHAR(20) PRIMARY KEY,
    Codigo_IATA_Aerolinea CHAR(2) NOT NULL REFERENCES Aerolineas(Codigo_IATA_Aerolinea) ON DELETE CASCADE,
    Nombre_Completo       VARCHAR(150) NOT NULL,
    Rol                   VARCHAR(50) NOT NULL
);
CREATE TABLE Pasajeros (
    Numero_Pasaporte VARCHAR(20) PRIMARY KEY,
    Nombre_Completo  VARCHAR(150) NOT NULL,
    Fecha_Nacimiento DATE NOT NULL,
    Nacionalidad     VARCHAR(50) NOT NULL
);
CREATE TABLE Servicios_Adicionales (
    Codigo_Servicio VARCHAR(10) PRIMARY KEY,
    Descripcion     VARCHAR(150) NOT NULL,
    Precio_Base     NUMERIC(10,2) NOT NULL
);
CREATE TABLE Vuelos (
    ID_Vuelo_Operacion    VARCHAR(20) PRIMARY KEY,
    Codigo_IATA_Aerolinea CHAR(2) NOT NULL REFERENCES Aerolineas(Codigo_IATA_Aerolinea),
    Origen_IATA           CHAR(3) NOT NULL REFERENCES Aeropuertos(Codigo_IATA_Aeropuerto),
    Destino_IATA          CHAR(3) NOT NULL REFERENCES Aeropuertos(Codigo_IATA_Aeropuerto),
    Codigo_Modelo         VARCHAR(15) NOT NULL REFERENCES Modelos_Avion(Codigo_Modelo),
    Fecha_Hora_Salida     TIMESTAMP NOT NULL,
    Fecha_Hora_Llegada    TIMESTAMP NOT NULL,
    Estado_Vuelo          VARCHAR(20) NOT NULL
);
CREATE INDEX idx_vuelos_fecha ON Vuelos(Fecha_Hora_Salida);
CREATE INDEX idx_vuelos_aero  ON Vuelos(Codigo_IATA_Aerolinea);
CREATE INDEX idx_vuelos_ruta  ON Vuelos(Origen_IATA, Destino_IATA);
CREATE TABLE Reservas (
    PNR_Localizador    CHAR(6) PRIMARY KEY,
    ID_Vuelo_Operacion VARCHAR(20) NOT NULL REFERENCES Vuelos(ID_Vuelo_Operacion) ON DELETE CASCADE,
    Fecha_Emision      TIMESTAMP NOT NULL,
    Estado_Pago        VARCHAR(20) NOT NULL,
    Monto_Total        NUMERIC(10,2) NOT NULL
);
CREATE INDEX idx_reservas_vuelo ON Reservas(ID_Vuelo_Operacion);
CREATE TABLE Aerolinea_Aeropuertos (
    Codigo_IATA_Aerolinea  CHAR(2) NOT NULL REFERENCES Aerolineas(Codigo_IATA_Aerolinea)  ON DELETE CASCADE,
    Codigo_IATA_Aeropuerto CHAR(3) NOT NULL REFERENCES Aeropuertos(Codigo_IATA_Aeropuerto) ON DELETE CASCADE,
    Es_Hub_Principal       BOOLEAN NOT NULL,
    Terminal_Asignada      VARCHAR(10),
    PRIMARY KEY (Codigo_IATA_Aerolinea, Codigo_IATA_Aeropuerto)
);
CREATE TABLE Tripulacion_Vuelo (
    ID_Vuelo_Operacion VARCHAR(20) NOT NULL REFERENCES Vuelos(ID_Vuelo_Operacion) ON DELETE CASCADE,
    Numero_Licencia    VARCHAR(20) NOT NULL REFERENCES Empleados(Numero_Licencia) ON DELETE CASCADE,
    Rol_En_Vuelo       VARCHAR(50) NOT NULL,
    PRIMARY KEY (ID_Vuelo_Operacion, Numero_Licencia)
);
CREATE INDEX idx_trip_vuelo ON Tripulacion_Vuelo(ID_Vuelo_Operacion);
CREATE TABLE Pasajeros_Reserva (
    PNR_Localizador  CHAR(6)     NOT NULL REFERENCES Reservas(PNR_Localizador)   ON DELETE CASCADE,
    Numero_Pasaporte VARCHAR(20) NOT NULL REFERENCES Pasajeros(Numero_Pasaporte) ON DELETE CASCADE,
    Asiento_Asignado VARCHAR(5),
    Clase_Tarifa     VARCHAR(30) NOT NULL,
    PRIMARY KEY (PNR_Localizador, Numero_Pasaporte)
);
CREATE TABLE Servicios_Reserva (
    PNR_Localizador      CHAR(6)     NOT NULL REFERENCES Reservas(PNR_Localizador)              ON DELETE CASCADE,
    Codigo_Servicio      VARCHAR(10) NOT NULL REFERENCES Servicios_Adicionales(Codigo_Servicio) ON DELETE CASCADE,
    Cantidad             INT NOT NULL,
    Costo_Final_Aplicado NUMERIC(10,2) NOT NULL,
    PRIMARY KEY (PNR_Localizador, Codigo_Servicio)
);
CREATE TABLE Boletos (
    Numero_Boleto      VARCHAR(13) PRIMARY KEY,
    PNR_Localizador    CHAR(6)     NOT NULL REFERENCES Reservas(PNR_Localizador) ON DELETE CASCADE,
    Numero_Pasaporte   VARCHAR(20) NOT NULL REFERENCES Pasajeros(Numero_Pasaporte),
    ID_Vuelo_Operacion VARCHAR(20) NOT NULL REFERENCES Vuelos(ID_Vuelo_Operacion),
    Fecha_Emision      TIMESTAMP NOT NULL,
    Asiento_Asignado   VARCHAR(5),
    Clase_Tarifa       VARCHAR(30) NOT NULL,
    Estado_Boleto      VARCHAR(20) NOT NULL
);
CREATE INDEX idx_boletos_vuelo ON Boletos(ID_Vuelo_Operacion);
CREATE INDEX idx_boletos_pnr   ON Boletos(PNR_Localizador);
CREATE INDEX idx_boletos_pas   ON Boletos(Numero_Pasaporte);
CREATE INDEX idx_boletos_fecha ON Boletos(Fecha_Emision);
"""
conn = psycopg2.connect(dbname=DB_NAME, **DB_CONFIG)
cur = conn.cursor(); cur.execute(DDL); conn.commit()
cur.close(); conn.close()
print('DDL: 13 tablas creadas')

DDL: 13 tablas creadas


## 3. Catálogos

In [5]:
AEROLINEAS = {
    'AM': ('Aeroméxico','México'),       'IB': ('Iberia','España'),
    'LH': ('Lufthansa','Alemania'),      'UA': ('United Airlines','Estados Unidos'),
    'AV': ('Avianca','Colombia'),        'AA': ('American Airlines','Estados Unidos'),
    'AF': ('Air France','Francia'),      'KL': ('KLM','Países Bajos'),
}
AEROPUERTOS = {
    'MEX': ('Aeropuerto Internacional Benito Juárez','Ciudad de México','México'),
    'CUN': ('Aeropuerto Internacional de Cancún','Cancún','México'),
    'GDL': ('Aeropuerto Internacional Miguel Hidalgo','Guadalajara','México'),
    'MAD': ('Aeropuerto Adolfo Suárez Madrid-Barajas','Madrid','España'),
    'BCN': ('Aeropuerto de Barcelona-El Prat','Barcelona','España'),
    'FRA': ('Aeropuerto de Fráncfort del Meno','Fráncfort','Alemania'),
    'JFK': ('Aeropuerto Internacional John F. Kennedy','Nueva York','Estados Unidos'),
    'LAX': ('Aeropuerto Internacional de Los Ángeles','Los Ángeles','Estados Unidos'),
    'MIA': ('Aeropuerto Internacional de Miami','Miami','Estados Unidos'),
    'CDG': ('Aeropuerto París-Charles de Gaulle','París','Francia'),
    'AMS': ('Aeropuerto de Ámsterdam-Schiphol','Ámsterdam','Países Bajos'),
    'BOG': ('Aeropuerto Internacional El Dorado','Bogotá','Colombia'),
}
MODELOS = {
    'B789': ('Boeing',274,14140), 'A35K': ('Airbus',350,15000),
    'A320': ('Airbus',150,6100),  'B738': ('Boeing',160,5765),
    'A321': ('Airbus',220,7400),  'B777': ('Boeing',396,9700),
}
SERVICIOS = {
    'WIFI01': ('Internet a bordo durante todo el vuelo',15.00),
    'LUGG23': ('Maleta documentada extra de 23 kg',45.00),
    'PRIO01': ('Abordaje prioritario',10.00),
    'PETS01': ('Transporte de mascota en cabina',80.00),
    'MEAL01': ('Menú gourmet a bordo',25.00),
    'SEAT01': ('Selección de asiento preferente',18.00),
    'LOUNG1': ('Acceso a sala VIP',55.00),
    'INSU01': ('Seguro de viaje básico',22.50),
}
CLASES        = ['Turista','Premier','Business','Primera']
ESTADO_VUELO  = ['Programado','En vuelo','Aterrizado','Retrasado','Cancelado']
ESTADO_PAGO   = ['Pagado','Pendiente','Reembolsado']
ESTADO_BOLETO = ['Emitido','Check-in','Abordado','Cancelado']
NOMBRES = ['Juan','José','Rosa','María','Luis','Ana','Carlos','Elena','Pedro','Sofía',
           'Diego','Carmen','Miguel','Laura','Andrés','Patricia','Jorge','Lucía',
           'Fernando','Isabel','Roberto','Daniela','Ricardo','Mónica']
APELLIDOS = ['Pérez','López','Santiago','García','Rodríguez','Martínez','Hernández',
             'Sánchez','Ramírez','Cruz','Gómez','Flores','Torres','Vargas','Castro',
             'Reyes','Romero','Morales','Ortega','Jiménez']
NACIONALIDADES = ['Mexicana','Española','Estadounidense','Colombiana','Francesa',
                  'Alemana','Neerlandesa','Polaca','Argentina','Brasileña','Italiana',
                  'Británica','Canadiense','Japonesa']
ROLES_TRIPULACION = ['Capitán','Primer Oficial','Sobrecargo Jefe','Sobrecargo']
print(f'Cat: {len(AEROLINEAS)} aero · {len(AEROPUERTOS)} aerop · {len(MODELOS)} mod · {len(SERVICIOS)} svc')

Cat: 8 aero · 12 aerop · 6 mod · 8 svc


In [6]:
conn = psycopg2.connect(dbname=DB_NAME, **DB_CONFIG)
cur = conn.cursor()

execute_values(cur, "INSERT INTO Aerolineas VALUES %s",
    [(k,v[0],v[1]) for k,v in AEROLINEAS.items()])
execute_values(cur, "INSERT INTO Aeropuertos VALUES %s",
    [(k,v[0],v[1],v[2]) for k,v in AEROPUERTOS.items()])
execute_values(cur, "INSERT INTO Modelos_Avion VALUES %s",
    [(k,v[0],v[1],v[2]) for k,v in MODELOS.items()])
execute_values(cur, "INSERT INTO Servicios_Adicionales VALUES %s",
    [(k,v[0],v[1]) for k,v in SERVICIOS.items()])

# Empleados: 8-10 por aerolínea → ~70 (necesarios para tripular ~4-5k vuelos sin agotar)
empleados = []
lic = 1
for cod_aero in AEROLINEAS:
    for _ in range(random.randint(8, 10)):
        nl = f'LIC{lic:06d}'; lic += 1
        nombre = f'{random.choice(NOMBRES)} {random.choice(APELLIDOS)} {random.choice(APELLIDOS)}'
        empleados.append((nl, cod_aero, nombre, random.choice(ROLES_TRIPULACION)))
execute_values(cur, "INSERT INTO Empleados VALUES %s", empleados)

# Pasajeros
N_PASAJEROS = 1500
pasajeros = []; vistos = set()
while len(pasajeros) < N_PASAJEROS:
    p = ''.join(random.choices(string.ascii_uppercase,k=2)) + ''.join(random.choices(string.digits,k=7))
    if p in vistos: continue
    vistos.add(p)
    pasajeros.append((p,
        f'{random.choice(NOMBRES)} {random.choice(APELLIDOS)} {random.choice(APELLIDOS)}',
        date(random.randint(1955,2010), random.randint(1,12), random.randint(1,28)),
        random.choice(NACIONALIDADES)))
execute_values(cur, "INSERT INTO Pasajeros VALUES %s", pasajeros, page_size=500)

# Aerolinea_Aeropuertos
aero_aerop = []
for cod_aero in AEROLINEAS:
    aps = random.sample(list(AEROPUERTOS), random.randint(4,7))
    hub = random.choice(aps)
    for ap in aps:
        aero_aerop.append((cod_aero, ap, ap==hub, f'T{random.randint(1,4)}'))
execute_values(cur, "INSERT INTO Aerolinea_Aeropuertos VALUES %s", aero_aerop)

conn.commit(); cur.close(); conn.close()
print(f'Insertados: {len(empleados)} empl · {len(pasajeros)} pas · {len(aero_aerop)} aero-aerop')

Insertados: 71 empl · 1500 pas · 44 aero-aerop


## 4. Generación masiva PL/pgSQL — 731 días completos

Recorre el rango entero (sin corte por volumen). 5-7 vuelos/día × 40-80 pasajeros → ~200k boletos. Pobla `Tripulacion_Vuelo` (2-3 empleados por vuelo, de la misma aerolínea).

In [7]:
PLPGSQL = r"""
DO $$
DECLARE
    v_fecha_ini  TIMESTAMP := '2023-01-01 00:00:00';
    v_fecha_fin  TIMESTAMP := '2024-12-31 23:59:59';
    v_dia        TIMESTAMP;
    v_n_vuelos   INT;
    v_id_vuelo   VARCHAR(20);
    v_pnr        CHAR(6);
    v_aero       CHAR(2);
    v_origen     CHAR(3);
    v_destino    CHAR(3);
    v_modelo     VARCHAR(15);
    v_salida     TIMESTAMP;
    v_llegada    TIMESTAMP;
    v_n_pas      INT;
    v_pasap      VARCHAR(20);
    v_clase      VARCHAR(30);
    v_asiento    VARCHAR(5);
    v_monto      NUMERIC(10,2);
    v_n_boleto   VARCHAR(13);
    v_estado_v   VARCHAR(20);
    v_estado_p   VARCHAR(20);
    v_estado_b   VARCHAR(20);
    v_pasap_set  TEXT[];
    v_emp_aero   TEXT[];
    v_n_trip     INT;
    v_lic        VARCHAR(20);
    v_rol_trip   VARCHAR(50);
    v_i          INT;
    v_j          INT;
    v_total_b    BIGINT := 0;
    v_total_v    BIGINT := 0;
    v_total_t    BIGINT := 0;
    v_aerolineas   CHAR(2)[]     := ARRAY['AM','IB','LH','UA','AV','AA','AF','KL'];
    v_aeropuertos  CHAR(3)[]     := ARRAY['MEX','CUN','GDL','MAD','BCN','FRA','JFK','LAX','MIA','CDG','AMS','BOG'];
    v_modelos      VARCHAR(15)[] := ARRAY['B789','A35K','A320','B738','A321','B777'];
    v_clases       VARCHAR(30)[] := ARRAY['Turista','Premier','Business','Primera'];
    v_estados_v    VARCHAR(20)[] := ARRAY['Programado','Aterrizado','Retrasado','Cancelado'];
    v_estados_p    VARCHAR(20)[] := ARRAY['Pagado','Pagado','Pagado','Pendiente','Reembolsado'];
    v_estados_b    VARCHAR(20)[] := ARRAY['Emitido','Check-in','Abordado','Cancelado'];
    v_servicios    VARCHAR(10)[] := ARRAY['WIFI01','LUGG23','PRIO01','PETS01','MEAL01','SEAT01','LOUNG1','INSU01'];
    v_roles_t      VARCHAR(50)[] := ARRAY['Capitán','Primer Oficial','Sobrecargo Jefe','Sobrecargo'];
BEGIN
    PERFORM setseed(0.42);
    SELECT ARRAY(SELECT Numero_Pasaporte FROM Pasajeros) INTO v_pasap_set;

    v_dia := v_fecha_ini;
    WHILE v_dia <= v_fecha_fin LOOP
        v_n_vuelos := 5 + floor(random() * 3)::INT;

        FOR v_i IN 1..v_n_vuelos LOOP
            v_aero := v_aerolineas[1 + floor(random() * array_length(v_aerolineas,1))::INT];
            v_origen := v_aeropuertos[1 + floor(random() * array_length(v_aeropuertos,1))::INT];
            LOOP
                v_destino := v_aeropuertos[1 + floor(random() * array_length(v_aeropuertos,1))::INT];
                EXIT WHEN v_destino <> v_origen;
            END LOOP;
            v_modelo   := v_modelos[1 + floor(random() * array_length(v_modelos,1))::INT];
            v_estado_v := v_estados_v[1 + floor(random() * array_length(v_estados_v,1))::INT];
            v_salida   := v_dia + (floor(random() * 1440)::INT || ' minutes')::INTERVAL;
            v_llegada  := v_salida + ((60 + floor(random() * 720)::INT) || ' minutes')::INTERVAL;
            v_id_vuelo := v_aero || lpad((v_total_v + 1)::TEXT, 6, '0') || '-' || to_char(v_dia,'YYYYMMDD');

            INSERT INTO Vuelos
              (ID_Vuelo_Operacion,Codigo_IATA_Aerolinea,Origen_IATA,Destino_IATA,Codigo_Modelo,Fecha_Hora_Salida,Fecha_Hora_Llegada,Estado_Vuelo)
            VALUES (v_id_vuelo,v_aero,v_origen,v_destino,v_modelo,v_salida,v_llegada,v_estado_v);
            v_total_v := v_total_v + 1;

            -- Tripulacion: 2-3 empleados de la misma aerolinea (sin duplicar)
            SELECT ARRAY(SELECT Numero_Licencia FROM Empleados WHERE Codigo_IATA_Aerolinea = v_aero) INTO v_emp_aero;
            v_n_trip := 2 + floor(random() * 2)::INT;
            FOR v_j IN 1..v_n_trip LOOP
                v_lic := v_emp_aero[1 + floor(random() * array_length(v_emp_aero,1))::INT];
                v_rol_trip := v_roles_t[1 + floor(random() * array_length(v_roles_t,1))::INT];
                INSERT INTO Tripulacion_Vuelo (ID_Vuelo_Operacion,Numero_Licencia,Rol_En_Vuelo)
                VALUES (v_id_vuelo, v_lic, v_rol_trip)
                ON CONFLICT DO NOTHING;
                v_total_t := v_total_t + 1;
            END LOOP;

            -- Reserva grupal
            v_pnr := upper(substr(md5(random()::TEXT || v_id_vuelo), 1, 6));
            v_n_pas := 40 + floor(random() * 41)::INT;
            v_monto := round((v_n_pas * (200 + random() * 800))::NUMERIC, 2);
            v_estado_p := v_estados_p[1 + floor(random() * array_length(v_estados_p,1))::INT];
            INSERT INTO Reservas VALUES (v_pnr, v_id_vuelo, v_salida - INTERVAL '15 days', v_estado_p, v_monto);

            INSERT INTO Servicios_Reserva VALUES (
                v_pnr,
                v_servicios[1 + floor(random() * array_length(v_servicios,1))::INT],
                1 + floor(random() * 3)::INT,
                round((10 + random() * 70)::NUMERIC, 2)
            ) ON CONFLICT DO NOTHING;

            FOR v_j IN 1..v_n_pas LOOP
                v_pasap := v_pasap_set[1 + floor(random() * array_length(v_pasap_set,1))::INT];
                v_clase := v_clases[1 + floor(random() * array_length(v_clases,1))::INT];
                v_asiento := (1 + floor(random() * 40))::TEXT || chr(65 + floor(random() * 6)::INT);
                v_estado_b := v_estados_b[1 + floor(random() * array_length(v_estados_b,1))::INT];
                v_n_boleto := '157' || lpad((v_total_b + 1)::TEXT, 10, '0');
                INSERT INTO Boletos VALUES
                  (v_n_boleto, v_pnr, v_pasap, v_id_vuelo, v_salida - INTERVAL '15 days', v_asiento, v_clase, v_estado_b);
                INSERT INTO Pasajeros_Reserva VALUES (v_pnr, v_pasap, v_asiento, v_clase) ON CONFLICT DO NOTHING;
                v_total_b := v_total_b + 1;
            END LOOP;
        END LOOP;

        v_dia := v_dia + INTERVAL '1 day';
    END LOOP;

    RAISE NOTICE 'Vuelos=% Boletos=% Tripulacion=%', v_total_v, v_total_b, v_total_t;
END $$;
"""

print('Ejecutando PL/pgSQL (2-5 min)...')
t0 = time.perf_counter()
conn = psycopg2.connect(dbname=DB_NAME, **DB_CONFIG); conn.autocommit = False
cur = conn.cursor(); cur.execute(PLPGSQL); conn.commit()
T_PLPGSQL = time.perf_counter() - t0
print(f'PL/pgSQL: {T_PLPGSQL:.2f} s')

CONTEOS = {}
for tbl in ['Vuelos','Reservas','Boletos','Pasajeros_Reserva','Servicios_Reserva','Tripulacion_Vuelo']:
    cur.execute(f'SELECT COUNT(*) FROM {tbl}')
    CONTEOS[tbl] = cur.fetchone()[0]
    print(f'  {tbl:<22}{CONTEOS[tbl]:>10,}')
cur.close(); conn.close()

Ejecutando PL/pgSQL (2-5 min)...
PL/pgSQL: 10.08 s
  Vuelos                     4,376
  Reservas                   4,376
  Boletos                  263,008
  Pasajeros_Reserva        257,804
  Servicios_Reserva          4,376
  Tripulacion_Vuelo          9,978


## 5. JSON jerárquico — 731 documentos

Misma información que el relacional, desnormalizada en árbol. Un documento por día.

```
fecha
 └ vuelos_del_dia[]
    ├ aerolinea {nombre, pais}
    ├ ruta {origen{...}, destino{...}}
    ├ aeronave {fabricante, modelo, capacidad}
    ├ horario {salida_iso, llegada_iso, estado}
    ├ tripulacion[] {nombre, rol, licencia}
    └ manifiesto_pasajeros[]
       ├ pasajero {nombre, nacionalidad, fecha_nacimiento_iso}
       ├ boleto {numero, clase, asiento, estado, pnr}
       └ servicios_contratados[] {descripcion, cantidad, costo}
```

In [8]:
conn = psycopg2.connect(dbname=DB_NAME, **DB_CONFIG)
cur = conn.cursor()

print('Extrayendo vuelos completos...')
cur.execute("""
    SELECT v.ID_Vuelo_Operacion,
           v.Fecha_Hora_Salida, v.Fecha_Hora_Llegada, v.Estado_Vuelo,
           ae.Nombre_Comercial, ae.Pais_Origen,
           ao.Nombre_Oficial, ao.Ciudad, ao.Pais,
           ad.Nombre_Oficial, ad.Ciudad, ad.Pais,
           ma.Fabricante, ma.Codigo_Modelo, ma.Capacidad_Pasajeros
    FROM Vuelos v
    JOIN Aerolineas ae    ON ae.Codigo_IATA_Aerolinea = v.Codigo_IATA_Aerolinea
    JOIN Aeropuertos ao   ON ao.Codigo_IATA_Aeropuerto = v.Origen_IATA
    JOIN Aeropuertos ad   ON ad.Codigo_IATA_Aeropuerto = v.Destino_IATA
    JOIN Modelos_Avion ma ON ma.Codigo_Modelo = v.Codigo_Modelo
    ORDER BY v.Fecha_Hora_Salida
""")
vuelos_rs = cur.fetchall()
print(f'  Vuelos: {len(vuelos_rs):,}')

print('Extrayendo tripulación...')
cur.execute("""
    SELECT t.ID_Vuelo_Operacion, e.Nombre_Completo, t.Rol_En_Vuelo, e.Numero_Licencia
    FROM Tripulacion_Vuelo t
    JOIN Empleados e ON e.Numero_Licencia = t.Numero_Licencia
""")
tripulacion_rs = cur.fetchall()
print(f'  Tripulación: {len(tripulacion_rs):,}')

print('Extrayendo boletos...')
cur.execute("""
    SELECT b.ID_Vuelo_Operacion,
           b.Numero_Boleto, b.Clase_Tarifa, b.Asiento_Asignado, b.Estado_Boleto, b.PNR_Localizador,
           p.Nombre_Completo, p.Nacionalidad, p.Fecha_Nacimiento
    FROM Boletos b
    JOIN Pasajeros p ON p.Numero_Pasaporte = b.Numero_Pasaporte
""")
boletos_rs = cur.fetchall()
print(f'  Boletos: {len(boletos_rs):,}')

print('Extrayendo servicios...')
cur.execute("""
    SELECT sr.PNR_Localizador, sa.Descripcion, sr.Cantidad, sr.Costo_Final_Aplicado
    FROM Servicios_Reserva sr
    JOIN Servicios_Adicionales sa ON sa.Codigo_Servicio = sr.Codigo_Servicio
""")
servicios_rs = cur.fetchall()
print(f'  Servicios: {len(servicios_rs):,}')

cur.close(); conn.close()

Extrayendo vuelos completos...
  Vuelos: 4,376
Extrayendo tripulación...
  Tripulación: 9,978
Extrayendo boletos...
  Boletos: 263,008
Extrayendo servicios...
  Servicios: 4,376


In [9]:
def _json_default(o):
    if isinstance(o, Decimal):  return float(o)
    if isinstance(o, datetime): return o.isoformat()
    if isinstance(o, date):     return o.isoformat()
    raise TypeError(f'No serializable: {type(o)}')

# Servicios por PNR
servicios_por_pnr = defaultdict(list)
for pnr, desc, cant, costo in servicios_rs:
    servicios_por_pnr[pnr].append({
        'descripcion': desc, 'cantidad': int(cant), 'costo_aplicado_usd': float(costo)
    })

# Tripulación por vuelo
tripulacion_por_vuelo = defaultdict(list)
for idv, nom, rol, lic in tripulacion_rs:
    tripulacion_por_vuelo[idv].append({
        'nombre_completo': nom, 'rol_en_vuelo': rol, 'numero_licencia': lic
    })

# Boletos por vuelo
boletos_por_vuelo = defaultdict(list)
for (idv, n_bol, clase, asiento, estado_b, pnr, nom_pas, nac, fnac) in boletos_rs:
    boletos_por_vuelo[idv].append({
        'pasajero': {
            'nombre_completo': nom_pas, 'nacionalidad': nac,
            'fecha_nacimiento_iso': fnac.isoformat(),
        },
        'boleto': {
            'numero_boleto': n_bol, 'clase_tarifa': clase,
            'asiento': asiento, 'estado': estado_b, 'localizador_pnr': pnr,
        },
        'servicios_contratados': servicios_por_pnr.get(pnr, []),
    })

# Agrupar vuelos por fecha
documentos_por_fecha = defaultdict(list)
for (idv, fsal, flleg, est_v, ae_nom, ae_pais,
     ao_nom, ao_ciu, ao_pais, ad_nom, ad_ciu, ad_pais,
     ma_fab, ma_mod, ma_cap) in vuelos_rs:
    fecha_key = fsal.date().isoformat()
    documentos_por_fecha[fecha_key].append({
        'identificador_operativo': idv,
        'aerolinea': {'nombre_comercial': ae_nom, 'pais_origen': ae_pais},
        'ruta': {
            'origen':  {'nombre_oficial': ao_nom, 'ciudad': ao_ciu, 'pais': ao_pais},
            'destino': {'nombre_oficial': ad_nom, 'ciudad': ad_ciu, 'pais': ad_pais},
        },
        'aeronave': {'fabricante': ma_fab, 'modelo': ma_mod, 'capacidad_pasajeros': int(ma_cap)},
        'horario': {
            'salida_iso':  fsal.isoformat(),
            'llegada_iso': flleg.isoformat(),
            'estado': est_v,
        },
        'tripulacion': tripulacion_por_vuelo.get(idv, []),
        'manifiesto_pasajeros': boletos_por_vuelo.get(idv, []),
    })

# Garantizar 731 documentos (uno por día del rango)
from datetime import timedelta
documentos = []
d = FECHA_INICIO.date()
fin = FECHA_FIN.date()
while d <= fin:
    fk = d.isoformat()
    vuelos = documentos_por_fecha.get(fk, [])
    documentos.append({
        'fecha': fk,
        'total_vuelos_dia': len(vuelos),
        'total_pasajeros': sum(len(v['manifiesto_pasajeros']) for v in vuelos),
        'vuelos_del_dia': vuelos,
    })
    d += timedelta(days=1)

assert len(documentos) == 731, f'Se esperaban 731 docs, hay {len(documentos)}'
print(f'Documentos JSON: {len(documentos)}')

# Serializar
out_json = os.path.join(OUTPUT_DIR, 'aeropuerto_jerarquico.json')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(documentos, f, ensure_ascii=False, default=_json_default)
size_mb = os.path.getsize(out_json) / 1024 / 1024
print(f'JSON guardado: {out_json}  ({size_mb:.1f} MB)')

Documentos JSON: 731
JSON guardado: ./parte1/aeropuerto_jerarquico.json  (101.1 MB)


In [10]:
def profundidad(obj, nivel=0):
    if isinstance(obj, dict):
        return max([profundidad(v, nivel+1) for v in obj.values()] or [nivel])
    if isinstance(obj, list) and obj:
        return max(profundidad(x, nivel+1) for x in obj)
    return nivel

# Usar un doc con vuelos para medir profundidad real
doc_con_vuelos = next(d for d in documentos if d['total_vuelos_dia'] > 0)
P_MAX = profundidad(doc_con_vuelos)
print(f'Profundidad máxima JSON: {P_MAX} niveles')
print(f'Total vuelos:   {sum(d["total_vuelos_dia"] for d in documentos):,}')
print(f'Total boletos:  {sum(d["total_pasajeros"] for d in documentos):,}')
print(f'Días con datos: {sum(1 for d in documentos if d["total_vuelos_dia"]>0)}/731')

Profundidad máxima JSON: 7 niveles
Total vuelos:   4,376
Total boletos:  263,008
Días con datos: 731/731


## 6. Segundo JSON multinivel — `transacciones_jerarquico.json`

Patrón "tipo Clases": raíz transaccional con **descripciones legibles** en lugar de códigos/FKs.
Jerarquía: **fecha → aerolínea (nombre) → ruta (origen→destino descriptivo) → pasajeros (nombres planos)**.
Reutiliza `vuelos_rs` y `boletos_rs` ya extraídos en la celda 12 (no re-consulta PG).

In [11]:
# Segundo JSON: patrón "tipo Clases" — descripciones en vez de FKs/códigos
# Reutiliza vuelos_rs y boletos_rs (celda 12). NO se re-consulta PostgreSQL.
from datetime import timedelta as _td

# Pasajeros por vuelo (sólo nombre, sin pasaporte ni boleto)
_pasajeros_por_vuelo = defaultdict(list)
for (idv, _n_bol, _clase, _asiento, _est_b, _pnr, nom_pas, _nac, _fnac) in boletos_rs:
    _pasajeros_por_vuelo[idv].append(nom_pas)

# Agrupar vuelos por (fecha, aerolinea_nombre, pais_aerolinea)
# y por ruta descriptiva dentro de cada aerolínea
_arbol = defaultdict(lambda: defaultdict(lambda: {'pais': None, 'rutas': []}))
# _arbol[fecha][aerolinea_nombre] = {'pais': str, 'rutas': [ {ruta, aeronave, pasajeros}, ... ]}

for (idv, fsal, _flleg, _est_v, ae_nom, ae_pais,
     ao_nom, ao_ciu, _ao_pais, ad_nom, ad_ciu, _ad_pais,
     ma_fab, ma_mod, ma_cap) in vuelos_rs:
    fecha_key = fsal.date().isoformat()
    nodo_ae = _arbol[fecha_key][ae_nom]
    nodo_ae['pais'] = ae_pais
    ruta_desc = f"{ao_ciu} → {ad_ciu}"
    aeronave_desc = f"{ma_fab} {ma_mod} ({int(ma_cap)} plazas)"
    nodo_ae['rutas'].append({
        'ruta': ruta_desc,
        'aeronave': aeronave_desc,
        'pasajeros': _pasajeros_por_vuelo.get(idv, []),
    })

# Construir array final, garantizando 731 días
transacciones = []
_d = FECHA_INICIO.date()
_fin = FECHA_FIN.date()
while _d <= _fin:
    fk = _d.isoformat()
    aerolineas_dia = []
    for ae_nom, datos in _arbol.get(fk, {}).items():
        aerolineas_dia.append({
            'aerolinea': ae_nom,
            'pais': datos['pais'],
            'rutas_operadas': datos['rutas'],
        })
    transacciones.append({
        'fecha': fk,
        'aerolineas_del_dia': aerolineas_dia,
    })
    _d += _td(days=1)

assert len(transacciones) == 731, f'Se esperaban 731 docs, hay {len(transacciones)}'

# Validar profundidad con la función existente sobre un día con datos
_doc_con_datos = next(d for d in transacciones if d['aerolineas_del_dia'])
P_TRANS = profundidad(_doc_con_datos)
assert P_TRANS >= 4, f'Profundidad insuficiente: {P_TRANS}'

# Serializar
out_trans = os.path.join(OUTPUT_DIR, 'transacciones_jerarquico.json')
with open(out_trans, 'w', encoding='utf-8') as f:
    json.dump(transacciones, f, ensure_ascii=False, default=_json_default)
size_trans_mb = os.path.getsize(out_trans) / 1024 / 1024

print(f'Documentos:    {len(transacciones)}')
print(f'Profundidad:   {P_TRANS} niveles (>=4 ✓)')
print(f'Tamaño:        {size_trans_mb:.2f} MB')
print(f'Guardado en:   {out_trans}')


Documentos:    731
Profundidad:   6 niveles (>=4 ✓)
Tamaño:        7.01 MB
Guardado en:   ./parte1/transacciones_jerarquico.json


## 7. Export a `parte1/` — artefactos para P2/P3/P4

In [12]:
# Conteos finales de TODAS las tablas
conn = psycopg2.connect(dbname=DB_NAME, **DB_CONFIG)
cur = conn.cursor()
TABLAS = ['Aerolineas','Aeropuertos','Modelos_Avion','Servicios_Adicionales',
          'Empleados','Pasajeros','Aerolinea_Aeropuertos','Vuelos','Reservas',
          'Pasajeros_Reserva','Servicios_Reserva','Boletos','Tripulacion_Vuelo']
conteos_final = {}
for t in TABLAS:
    cur.execute(f'SELECT COUNT(*) FROM {t}')
    conteos_final[t] = cur.fetchone()[0]
cur.close(); conn.close()

# catalogos.json — referencia para P2/P4
catalogos = {
    'aerolineas':   [{'iata':k,'nombre':v[0],'pais':v[1]} for k,v in AEROLINEAS.items()],
    'aeropuertos':  [{'iata':k,'nombre':v[0],'ciudad':v[1],'pais':v[2]} for k,v in AEROPUERTOS.items()],
    'modelos':      [{'codigo':k,'fabricante':v[0],'capacidad':v[1],'alcance_km':v[2]} for k,v in MODELOS.items()],
    'servicios':    [{'codigo':k,'descripcion':v[0],'precio_base':v[1]} for k,v in SERVICIOS.items()],
    'clases':       CLASES,
    'estado_vuelo': ESTADO_VUELO,
    'estado_pago':  ESTADO_PAGO,
    'estado_boleto':ESTADO_BOLETO,
}
with open(os.path.join(OUTPUT_DIR,'catalogos.json'),'w',encoding='utf-8') as f:
    json.dump(catalogos, f, ensure_ascii=False, indent=2)

# metricas_parte1.csv
with open(os.path.join(OUTPUT_DIR,'metricas_parte1.csv'),'w',newline='',encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['metrica','valor'])
    w.writerow(['tiempo_plpgsql_s', f'{T_PLPGSQL:.3f}'])
    w.writerow(['profundidad_json', P_MAX])
    w.writerow(['documentos_json',  len(documentos)])
    w.writerow(['json_size_mb',     f'{size_mb:.2f}'])
    for t,n in conteos_final.items():
        w.writerow([f'count_{t.lower()}', n])

# resumen_parte1.md
resumen = f"""# Resumen PARTE 1 — Modelado + JSON Jerárquico

## Universo de datos generado
- Rango: **2023-01-01 → 2024-12-31** (731 días, 100% cubiertos).
- Vuelos: **{conteos_final['Vuelos']:,}** · Boletos: **{conteos_final['Boletos']:,}** · Tripulación: **{conteos_final['Tripulacion_Vuelo']:,}**
- JSON: {len(documentos)} documentos · profundidad {P_MAX} niveles · {size_mb:.1f} MB.
- PL/pgSQL: {T_PLPGSQL:.2f} s.

## Decisiones que afectan P2/P3/P4

1. **Mismo universo en relacional y JSON.** El JSON es la desnormalización exacta del relacional (joins implícitos como árbol). P2 (Mongo vs SQL), P3 (BigQuery vs SQL) y P4 (Neo4j vs SQL) deben consumir este mismo dataset.
2. **Fechas como ISO `YYYY-MM-DDTHH:MM:SS`** sin timezone, idénticas al TIMESTAMP de PG. **P3 BigQuery: ingerir tal cual, sin reformatear** (causa raíz del bug original).
3. **`fecha_nacimiento_iso`** en JSON (no `edad` precomputada) → determinismo. Si P3/P4 necesitan edad, calcular contra fecha del vuelo.
4. **`Tripulacion_Vuelo` poblada** ({conteos_final['Tripulacion_Vuelo']:,} filas, 2-3 empleados/vuelo de la misma aerolínea). **P4 lo usa** para traversal Empleado→Vuelo→Aerolínea, dando nivel extra de profundidad vs SQL.
5. **Semillas fijas:** `random.seed(42)` Python, `setseed(0.42)` PG.
6. **`Decimal` → `float`** en JSON (función `_json_default`). Mongo y BigQuery deben tolerar floats.

## Artefactos exportados a `parte1/`
- `aeropuerto_jerarquico.json` — 731 docs (P2 Mongo · P3 BigQuery).
- `catalogos.json` — referencia rápida (P2/P4).
- `metricas_parte1.csv` — números para el reporte final.
- `resumen_parte1.md` — este archivo.

## Conteos finales por tabla
| Tabla | Filas |
|---|---:|
"""
for t,n in conteos_final.items():
    resumen += f"| {t} | {n:,} |\n"

with open(os.path.join(OUTPUT_DIR,'resumen_parte1.md'),'w',encoding='utf-8') as f:
    f.write(resumen)

print(f'\nArtefactos en {OUTPUT_DIR}/:')
for fn in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, fn)
    print(f'  {fn:<35} {os.path.getsize(p)/1024:>10.1f} KB')


Artefactos en ./parte1/:
  aeropuerto_jerarquico.json            103478.0 KB
  catalogos.json                             4.4 KB
  dataset_manifest.json                      0.3 KB
  metricas_parte1.csv                        0.4 KB
  resumen_parte1.md                          1.8 KB
  transacciones_jerarquico.json           7176.3 KB


## 8. SHA-256 del dataset canónico — `dataset_manifest.json`

Hash del archivo canónico (`aeropuerto_jerarquico.json`) para que P2/P3/P4 verifiquen integridad del dataset compartido.

In [13]:
# SHA-256 del dataset canónico + manifest
import hashlib

canon_path = os.path.join(OUTPUT_DIR, 'aeropuerto_jerarquico.json')

_h = hashlib.sha256()
with open(canon_path, 'rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        _h.update(chunk)
sha = _h.hexdigest()
tamano = os.path.getsize(canon_path)

manifest = {
    'archivo_canonico': 'aeropuerto_jerarquico.json',
    'sha256':           sha,
    'tamano_bytes':     tamano,
    'n_documentos':     len(documentos),
    'rango':            {'inicio': FECHA_INICIO.date().isoformat(),
                         'fin':    FECHA_FIN.date().isoformat()},
    'generado_en':      datetime.now().isoformat(timespec='seconds'),
}

manifest_path = os.path.join(OUTPUT_DIR, 'dataset_manifest.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f'SHA-256:  {sha}')
print(f'Tamaño:   {tamano:,} bytes')
print(f'Docs:     {manifest["n_documentos"]}')
print(f'Manifest: {manifest_path}')


SHA-256:  454163d8e4c2276f4446a0408165b663ec6656ecdbb1e9a2b770538b78fef442
Tamaño:   105,961,482 bytes
Docs:     731
Manifest: ./parte1/dataset_manifest.json


In [14]:
# Export demo.json — primer documento del JSON canónico para inspección rápida
demo_path = os.path.join(OUTPUT_DIR, 'demo.json')

with open(demo_path, 'w', encoding='utf-8') as f:
    json.dump(documentos[0], f, ensure_ascii=False, indent=2, default=_json_default)

size_kb = os.path.getsize(demo_path) / 1024
print(f'Demo guardado: {demo_path}  ({size_kb:.1f} KB)')
print(f'Fecha del documento: {documentos[0]["fecha"]}')
print(f'Vuelos del día:      {documentos[0]["total_vuelos_dia"]}')
print(f'Pasajeros del día:   {documentos[0]["total_pasajeros"]}')

Demo guardado: ./parte1/demo.json  (271.2 KB)
Fecha del documento: 2023-01-01
Vuelos del día:      7
Pasajeros del día:   422
